# Coursework 1 Group (text)


Group number: 9

Student names and k-numbers:
1. Henry Longe - k2552174
2. Nsetu Tarimo - k2604076
3. Thaw Zin Lin Myat - k2543493
4. Henry Akwarandu - k2557070

# Load modules (code)

In [12]:
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
import plotly.graph_objects as go
import plotly.figure_factory as ff
from sklearn.model_selection import train_test_split

#sklearn classification models 
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.svm import SVC
# from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier


#Pipeline
from sklearn.pipeline import Pipeline

#preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import label_binarize

#Dimensionality Reduction
from sklearn.decomposition import PCA

#Hyperparameter Tuning
from sklearn.model_selection import GridSearchCV

#Evaluation Metrics
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

#Confusion Matrix
from sklearn.metrics import confusion_matrix





# Load data

In [13]:
#data load
from sklearn.datasets import fetch_openml

dataset = fetch_openml(data_id=4538, as_frame=False) # fetch the dataset

## Checking Integrity and Train Test Split

In [14]:
# X and y
X = dataset.data
y = dataset.target


print("The shape of the Features are: ", X.shape)
print("The shape of the Target is: ",y.shape)

#checking integrity of data
distribution = np.unique(y, return_counts=True)
isnan = np.isnan(X).sum()

print("General distribution of data :", distribution)
print("Number of Null values (Nans) in dataset: ", isnan)


The shape of the Features are:  (9873, 32)
The shape of the Target is:  (9873,)
General distribution of data : (array(['D', 'H', 'P', 'R', 'S'], dtype=object), array([2741,  998, 2097, 1087, 2950]))
Number of Null values (Nans) in dataset:  0


### Isolation Outlier Detection

In [15]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(
    contamination=0.05,
    random_state=42
)

outlier_labels = iso.fit_predict(X)
# -1 = outlier, 1 = inlier

n_outliers = (outlier_labels == -1).sum()
print("Detected outliers:", n_outliers)

Detected outliers: 494


### Train Test Split


In [16]:
#train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, test_size=0.3, stratify=y, random_state=42)
print(f"The shape of the training dataset is: Features: {X_train.shape} Target: {np.unique(y_train)}")

print(f"The shape of the testing dataset is: Features: {X_test.shape} Target: {np.unique(y_test)}")
le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)
print(f"The following training target labels {np.unique(y_train)} have been encoded into {np.unique(y_train_enc)}")
print(f"The following testing target labels {np.unique(y_test)} have been encoded into {np.unique(y_test_enc)}")

The shape of the training dataset is: Features: (6911, 32) Target: ['D' 'H' 'P' 'R' 'S']
The shape of the testing dataset is: Features: (2962, 32) Target: ['D' 'H' 'P' 'R' 'S']
The following training target labels ['D' 'H' 'P' 'R' 'S'] have been encoded into [0 1 2 3 4]
The following testing target labels ['D' 'H' 'P' 'R' 'S'] have been encoded into [0 1 2 3 4]


# Classification

## Classification methods used (text)

1. Logistic Regression without PCA `cite`: https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
2. Decision Tree with PCA `cite`: https://scikit-learn.org/stable/modules/tree.html
3. Random Forest without PCA `cite`: https://scikit-learn.org/stable/modules/ensemble.html
4. Gradient Boost with PCA `cite`: https://scikit-learn.org/stable/modules/ensemble.html
5. Linear SVM without PCA `cite`: https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html
6. RBF SVM with PCA `cite`: https://scikit-learn.org/stable/api/sklearn.svm.html
7. KNN without PCA `cite`: https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html
8. Adaptive Boost with PCA `cite`: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html

## Training (code)

### Tuning Hyperparameters

In [6]:
#hyperparameter tuning with and finding out the best hyper parameter
models_tuning = {
    "Logistic Regression": (
        Pipeline([
            ("scaler", RobustScaler()),
            ("lg", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "lg__C": [0.001, 0.01, 0.1, 1, 10,100],
            "lg__solver": ['lbfgs', 'saga', 'newton-cg']
        }
    ),
    "Decision Tree (PCA)": (
        Pipeline([
            ("pca", PCA()),
            ("dt", DecisionTreeClassifier(
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "pca__n_components": [5 ,10 ,15 ,20],
            "dt__max_depth": [None ,1 , 5 , 10, 20],
            "dt__min_samples_split": [2, 5, 10],
            "dt__min_samples_leaf": [1, 5, 10]
        }   
    ),
    
    "Random Forest": (
        Pipeline([
            ("rf", RandomForestClassifier(
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "rf__n_estimators": [200, 500],
            "rf__max_depth": [None,3,5,7],
            "rf__min_samples_leaf": [5, 10, 15]
        }
        
    ), 
    "Gradient Boost (PCA)": (
        Pipeline([
            ("pca", PCA()),
            ("gb", GradientBoostingClassifier(random_state=42))
        ]),
        {
            "pca__n_components": [5,10,15,20],
            "gb__n_estimators": [200,400],
            "gb__learning_rate": [0.01,0.1,1],
            "gb__max_depth": [5,10,15],
            "gb__subsample": [0.8,1.0]
        }
    ),
    
    "Linear SVC": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LinearSVC(
                class_weight="balanced",
                max_iter=5000
            ))
        ]),
        {
            "clf__C": [1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100], 
            "clf__loss": ["squared_hinge"],
            "clf__dual": [False] 
        }
    ),
    "SVC [RBF] (PCA)": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA()),
            ("clf", SVC(
                kernel="rbf",
                class_weight="balanced",
                probability=False
            ))
        ]),
        {
            "pca__n_components": [5 ,10 ,15 ,20],
            "clf__C": [0.1, 1, 10, 100],
            "clf__gamma": [0.001, 0.01, 0.1]
        }
    ),
    "KNN": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(
                algorithm="brute",
            ))
        ]),
        {
            "clf__n_neighbors": list(range(1, 51, 2)),
            "clf__weights": ["uniform", "distance"],
            "clf__metric": ["euclidean", "manhattan"]
        }
    ),
    "Adaboost (PCA)": (
        Pipeline([
            ("pca", PCA()),
            ("ada", AdaBoostClassifier(random_state=42))
        ]),
        {
            "pca__n_components": [5,10,15,20],
            "ada__n_estimators": [200, 400, 500],
            "ada__learning_rate": [0.01, 0.1, 1],
        }
    )
}


tuned_results = {}
best_models = {}
best_params ={}

for name, (model, params) in models_tuning.items():
    grid = GridSearchCV(
        model,
        params,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1
    )
    
    grid.fit(X_train, y_train_enc)    
    
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    acc = balanced_accuracy_score(y_test_enc, y_pred)
    
    tuned_results[name] = acc
    best_models[name] = best_model
    best_params[name] = grid.best_params_



In [7]:
# Print Tuned Results for each different model
print("\nTuned Results (Balanced Accuracy):")
for k, v in tuned_results.items():
    print(f"{k}: {v:.4f}")


Tuned Results (Balanced Accuracy):
Logistic Regression: 0.4363
Decision Tree (PCA): 0.4814
Random Forest: 0.6045
Gradient Boost (PCA): 0.5982
Linear SVC: 0.4085
SVC [RBF] (PCA): 0.4985
KNN: 0.6354
Adaboost (PCA): 0.3784


In [19]:
# Print Best Models for each different model
print("\nBest Model:")
for k, v in best_models.items():
    print(f"{k}: {v}")


Best Model:
Logistic Regression: Pipeline(steps=[('scaler', RobustScaler()),
                ('lg',
                 LogisticRegression(C=0.1, class_weight='balanced',
                                    max_iter=2000, random_state=42,
                                    solver='newton-cg'))])
Decision Tree (PCA): Pipeline(steps=[('pca', PCA(n_components=10)),
                ('dt',
                 DecisionTreeClassifier(class_weight='balanced', max_depth=10,
                                        random_state=42))])
Random Forest: Pipeline(steps=[('rf',
                 RandomForestClassifier(class_weight='balanced',
                                        min_samples_leaf=5, n_estimators=500,
                                        random_state=42))])
Gradient Boost (PCA): Pipeline(steps=[('pca', PCA(n_components=20)),
                ('gb',
                 GradientBoostingClassifier(max_depth=15, n_estimators=400,
                                            random_state=42, subs

In [20]:
# Print Best Hyperparameters for each different model
print("\nBest Hyperparameter:")
for k, v in best_params.items():
    print(f"{k}: {v}")


Best Hyperparameter:
Logistic Regression: {'lg__C': 0.1, 'lg__solver': 'newton-cg'}
Decision Tree (PCA): {'dt__max_depth': 10, 'dt__min_samples_leaf': 1, 'dt__min_samples_split': 2, 'pca__n_components': 10}
Random Forest: {'rf__max_depth': None, 'rf__min_samples_leaf': 5, 'rf__n_estimators': 500}
Gradient Boost (PCA): {'gb__learning_rate': 0.1, 'gb__max_depth': 15, 'gb__n_estimators': 400, 'gb__subsample': 0.8, 'pca__n_components': 20}
Linear SVC: {'clf__C': 10, 'clf__dual': False, 'clf__loss': 'squared_hinge'}
SVC [RBF] (PCA): {'clf__C': 10, 'clf__gamma': 0.1, 'pca__n_components': 15}
KNN: {'clf__metric': 'manhattan', 'clf__n_neighbors': 1, 'clf__weights': 'uniform'}
Adaboost (PCA): {'ada__learning_rate': 1, 'ada__n_estimators': 500, 'pca__n_components': 20}


### Hypertuned models

In [17]:
models = {
    #  Logistic Regression 
    "Logistic Regression": (
        Pipeline([
            ("scaler", RobustScaler()),
            ("lr", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "lr__C":0.1,
            "lr__solver": 'newton-cg',
        }
    ),

    #  Decision Tree 
    "Decision Tree (PCA)": (
        Pipeline([
            ("pca", PCA()),
            ("dt", DecisionTreeClassifier(
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "dt__max_depth": 10,
            "dt__min_samples_leaf": 1,
            "dt__min_samples_split": 2,
            "pca__n_components": 10,
        }
    ),

    #  Random Forest 
    "Random Forest": (
        Pipeline([
            ("rf", RandomForestClassifier(
                n_estimators=500,
                max_depth=None,
                min_samples_leaf=5,
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "rf__n_estimators":500,
            "rf__max_depth": None,
            "rf__min_samples_leaf": 5,
        }
    ),

    #  Gradient Boost
    "Gradient Boost (PCA)": (
        Pipeline([
            ("pca", PCA()),
            ("gb", GradientBoostingClassifier(
                random_state=42
            ))
        ]),
        {
            "gb__n_estimators": 400,
            "gb__learning_rate": 0.1,
            "gb__max_depth": 15,
            "gb__subsample":0.8,
            "pca__n_components": 20,
        }
    ),

    #  Linear SVC 
    "Linear SVC": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("lsvm", LinearSVC(
                loss="squared_hinge",
                dual=False,
                class_weight="balanced",
                max_iter=5000,
                random_state=42
            ))
        ]),
        {
            "lsvm__C": 10,
            "lsvm__loss": "squared_hinge",
        }
    ),

    #  SVC RBF 
    "SVC (RBF) (PCA)": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA()),
            ("rbf", SVC(
                kernel="rbf",
                class_weight="balanced",
                probability=False,           
                random_state=42

            ))
        ]),
        {
            "rbf__C": 10,
            "rbf__gamma": 0.1,  
            "pca__n_components": 15,
        }
    ),

    #  KNN 
    "KNN": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("knn", KNeighborsClassifier(
                weights="uniform",
                algorithm="brute",             
            ))
        ]),
        {
            "knn__n_neighbors": 1,
            "knn__metric": "manhattan",
        }
    ),
    "Adaboost (PCA)": (
        Pipeline([
            ("pca", PCA()),
            ("ada", AdaBoostClassifier(random_state=42))
        ]),
        {
            "pca__n_components": 20,
            "ada__n_estimators": 500,
            "ada__learning_rate": 1,
        }
    ),
}

## Evaluation (code)

### Evaluate function

In [ ]:
def evaluate_models(models, X_train, X_test, y_train, y_test, label_encoder=None):

    encoded_classes = np.unique(y_train)

    if label_encoder is not None:
        classes = label_encoder.classes_  
    else:
        classes = encoded_classes
    
    n_classes = len(np.unique(y_train))
    y_test_bin = label_binarize(y_test, classes=np.unique(y_train))

    results = {}
    metrics = {}
    macro_rocs = {}
    micro_rocs = {}

    for name, (pipeline, params) in models.items():

        # Set parameters
        if params:
            pipeline.set_params(**params)

        # Wrap in OneVsRest for ROC computation
        clf = pipeline
        clf.fit(X_train, y_train)

        y_pred = clf.predict(X_test)
      
        # Balanced Accuracy
        bal_acc = balanced_accuracy_score(y_test, y_pred)

        # Get probability
        if hasattr(clf, "predict_proba"):
            y_score = clf.predict_proba(X_test)
        else:
            y_score = clf.decision_function(X_test)

        #Accuracy Score
        acc = accuracy_score(y_test, y_pred)

        #Recall Score
        micro_rec = recall_score(y_test, y_pred, average= 'micro')
        macro_rec = recall_score(y_test, y_pred, average= 'macro')

        #Precision Score
        micro_pre = precision_score(y_test, y_pred, average= 'micro', zero_division=0)
        macro_pre = precision_score(y_test, y_pred, average= 'macro', zero_division=0)

        #F1 Score
        micro_f1 = f1_score(y_test, y_pred, average= 'micro')
        macro_f1 = f1_score(y_test, y_pred, average= 'macro')

        # ROC AUC
        macro_auc = roc_auc_score(y_test_bin, y_score, average="macro")
        micro_auc = roc_auc_score(y_test_bin, y_score, average="micro")

        # #Just some Prints
        # print(f"Balanced Accuracy: {bal_acc:.4f}")
        # print(f"Macro-average OvR ROC AUC: {macro_auc:.4f}")
        # print(f"Micro-average OvR ROC AUC: {micro_auc:.4f}")

        # ---- Micro ROC ----
        fpr_micro, tpr_micro, _ = roc_curve(
            y_test_bin.ravel(), y_score.ravel()
        )

        per_class = {}
        fpr_dict = {}
        tpr_dict = {}

        for i in range(n_classes):
            fpr_i, tpr_i, _ = roc_curve(
                y_test_bin[:, i], y_score[:, i]
            )
            per_class[i] = (fpr_i, tpr_i)
            fpr_dict[i] = fpr_i
            tpr_dict[i] = tpr_i

        # ---- Macro ROC ----
        all_fpr = np.unique(
            np.concatenate([fpr_dict[i] for i in range(n_classes)])
        )

        mean_tpr = np.zeros_like(all_fpr)
        for i in range(n_classes):
            mean_tpr += np.interp(all_fpr, fpr_dict[i], tpr_dict[i])
        mean_tpr /= n_classes

        macro_rocs[name] = (all_fpr, mean_tpr)
        micro_rocs[name] = (fpr_micro,tpr_micro)

        #Confusion Matrix
        cm = confusion_matrix(y_test, y_pred, labels=encoded_classes)
        
        #results
        results[name] = {
            "balanced_accuracy": bal_acc,
            "accuracy": acc,
            "micro_recall": micro_rec, 
            "macro_recall": macro_rec, 
            "micro_precision": micro_pre, 
            "macro_precision": macro_pre, 
            "micro_f1": micro_f1, 
            "macro_f1": macro_f1, 
            "macro_auc": macro_auc,
            "micro_auc": micro_auc,
            "confusion_matrix": cm,
            "roc": {
                "macro": macro_rocs[name],
                "micro": micro_rocs[name],
                "per_class": per_class
            }
        }
        
        metrics[name] ={
            "balanced_accuracy": bal_acc,
            "accuracy": acc,
            "micro_recall": micro_rec, 
            "macro_recall": macro_rec, 
            "micro_precision": micro_pre, 
            "macro_precision": macro_pre, 
            "micro_f1": micro_f1, 
            "macro_f1": macro_f1, 
            "macro_auc": macro_auc,
            "micro_auc": micro_auc,
        }
    # =============================
    # Summary Table
    # =============================
    print("\n" + "="*80)
    print("Model PERFORMANCE SUMMARY")
    print("="*80)

    for name, result_metrics in results.items():
        print(f"{name}")
        print(f" Balanced Accuracy: {result_metrics['balanced_accuracy']:.4f}")
        print(f" Accuracy: {result_metrics['accuracy']:.4f}")
        print(f" Micro Recall: {result_metrics['micro_recall']:.4f}")
        print(f" Macro Recall: {result_metrics['macro_recall']:.4f}")
        print(f" Micro Precision: {result_metrics['micro_precision']:.4f}")
        print(f" Macro Precision: {result_metrics['macro_precision']:.4f}")
        print(f" Micro F1: {result_metrics['micro_f1']:.4f}")
        print(f" Macro-average ROC AUC: {result_metrics['macro_auc']:.4f}")
        print(f" Micro-average ROC AUC: {result_metrics['micro_auc']:.4f}")
        print("-"*60)
    return results, classes, metrics

### Plot Functions

In [19]:
def plot_macro_roc(results):

    fig = go.Figure()

    for name, res in results.items():
        fpr, tpr = res["roc"]["macro"]
        auc = res["macro_auc"]

        fig.add_trace(go.Scatter(
            x=fpr,
            y=tpr,
            mode="lines",
            name=f"{name} (AUC={auc:.3f})"
        ))

    fig.add_trace(go.Scatter(
        x=[0, 1], 
        y=[0, 1],
        mode="lines",
        line=dict(dash="dash"),
        name="Random"
    ))

    fig.update_layout(
        title="Macro-average One-vs-Rest ROC Curves",
        xaxis_title="False Positive Rate",
        yaxis_title="True Positive Rate"
    )

    fig.show()

In [20]:
def plot_micro_roc(results):

    fig = go.Figure()

    for name, res in results.items():
        fpr, tpr = res["roc"]["micro"]
        auc = res["micro_auc"]

        fig.add_trace(go.Scatter(
            x=fpr,
            y=tpr,
            mode="lines",
            name=f"{name} (AUC={auc:.3f})"
        ))

    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1],
        mode="lines",
        line=dict(dash="dash"),
        name="Random"
    ))

    fig.update_layout(
        title="Micro-average One-vs-Rest ROC Curves",
        xaxis_title="False Positive Rate",
        yaxis_title="True Positive Rate"
    )

    fig.show()

In [21]:
def plot_per_class_roc(results, classes):

    for i, cls in enumerate(classes):
        fig = go.Figure()

        for name, res in results.items():
            fpr, tpr = res["roc"]["per_class"][i]
            fig.add_trace(go.Scatter(
                x=fpr,
                y=tpr,
                mode="lines",
                name=name
            ))

        fig.add_trace(go.Scatter(
            x=[0, 1], y=[0, 1],
            mode="lines",
            line=dict(dash="dash"),
            name="Random"
        ))

        fig.update_layout(
            title=f"One-vs-Rest ROC Curves – Class {cls}",
            xaxis_title="False Positive Rate",
            yaxis_title="True Positive Rate"
        )

        fig.show()

In [22]:
def plot_confusion_matrices(results, classes):

    for name, res in results.items():
        cm = res["confusion_matrix"]

        fig = ff.create_annotated_heatmap(
            z=cm,
            x=[f"Pred {c}" for c in classes],
            y=[f"True {c}" for c in classes],
            colorscale="Blues"
        )

        fig.update_layout(
            title=f"Confusion Matrix – {name}",
            xaxis=dict(title="Predicted Label", side="bottom"),
            yaxis=dict(title="True Label")
        )

        fig.show()

### Results

In [23]:
results, classes, metrics = evaluate_models(
    models, X_train, X_test, y_train_enc, y_test_enc, label_encoder=le
)


Model PERFORMANCE SUMMARY
Logistic Regression
  Balanced Accuracy: 0.4363
 Accuracy: 0.4406
 Micro Recall: 0.4406
 Macro Recall: 0.4363
 Micro Precision: 0.4406
 Macro Precision: 0.4262
 Micro F1: 0.4406
 Macro-average ROC AUC: 0.7580
 Micro-average ROC AUC: 0.7546
------------------------------------------------------------
Decision Tree (PCA)
  Balanced Accuracy: 0.4814
 Accuracy: 0.4730
 Micro Recall: 0.4730
 Macro Recall: 0.4814
 Micro Precision: 0.4730
 Macro Precision: 0.4626
 Micro F1: 0.4730
 Macro-average ROC AUC: 0.7409
 Micro-average ROC AUC: 0.7591
------------------------------------------------------------
Random Forest
  Balanced Accuracy: 0.6045
 Accuracy: 0.6313
 Micro Recall: 0.6313
 Macro Recall: 0.6045
 Micro Precision: 0.6313
 Macro Precision: 0.6073
 Micro F1: 0.6313
 Macro-average ROC AUC: 0.8774
 Micro-average ROC AUC: 0.8887
------------------------------------------------------------
Gradient Boost (PCA)
  Balanced Accuracy: 0.5982
 Accuracy: 0.6671
 Micro Re

In [24]:
table_metrics = pd.DataFrame(metrics)
print("Performance Metrics") 
print(table_metrics)

Performance Metrics
                   Logistic Regression  Decision Tree (PCA)  Random Forest  \
balanced_accuracy             0.436341             0.481435       0.604517   
accuracy                      0.440581             0.472991       0.631330   
micro_recall                  0.440581             0.472991       0.631330   
macro_recall                  0.436341             0.481435       0.604517   
micro_precision               0.440581             0.472991       0.631330   
macro_precision               0.426232             0.462605       0.607254   
micro_f1                      0.440581             0.472991       0.631330   
macro_f1                      0.409769             0.451868       0.602325   
macro_auc                     0.758000             0.740902       0.877387   
micro_auc                     0.754603             0.759127       0.888672   

                   Gradient Boost (PCA)  Linear SVC  SVC (RBF) (PCA)  \
balanced_accuracy              0.598212    0.4084

### Plot Results

In [25]:
plot_macro_roc(results)
plot_micro_roc(results)

In [26]:
plot_per_class_roc(results, classes)

In [27]:
plot_confusion_matrices(results, classes)

# References (text)

1. Logistic Regression without PCA `cite`: https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
2. Decision Tree with PCA `cite`: https://scikit-learn.org/stable/modules/tree.html
3. Random Forest without PCA `cite`: https://scikit-learn.org/stable/modules/ensemble.html
4. Gradient Boost with PCA `cite`: https://scikit-learn.org/stable/modules/ensemble.html
5. Linear SVM without PCA `cite`: https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html
6. RBF SVM with PCA `cite`: https://scikit-learn.org/stable/api/sklearn.svm.html
7. KNN without PCA `cite`: https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html
8. Adaptive Boost with PCA `cite`: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html